# LangChain + Clembench experiment constructor

**Single configurable notebook** for running LangChain agents on clembench games.

**Only Cell 2 (Config) needs to be changed between experiments.**

| Variable | Description |
|---|---|
| `GAME` | any clembench game (e.g., `taboo`, `referencegame`, etc.|
| `MODEL` | any model (e.g.,`qwen`, `gpt-4o-mini`) |
| `AGENT_TYPE` | a LangChain agent (choose from below or create a custom one) |
| `RUN_ID` | any string — used as the results folder name |
| `NUM_EPISODES` | integer |
| `SINGLE_PASS` | `True` = one pass through instances, `False` = cycle infinitely |

In [ ]:
# ----- CONFIG ---------------------------------------------
GAME         = ""   
MODEL        = ""        
AGENT_TYPE   = ""  
RUN_ID       = ""
NUM_EPISODES = 26
SINGLE_PASS  = True
# ----------------------------------------------------------

## 1. Preparation

In [ ]:
import json
import httpx
import os

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from playpen.agents import ClemAgent, ClemObservation
from clemcore.clemgame import env, episode_results_folder_callbacks

from clemcore.backends import ModelRegistry
from clemcore.backends import KeyRegistry

In [ ]:
CLEMBENCH_HOME = r"path-to-clembench"
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [ ]:
# uncomment and run once to install dependencies, then comment out again
# %pip install -r $CLEMBENCH_HOME/requirements.txt
# %pip install --upgrade ipywidgets jupyter_client clemcore

# Make tqdm usable in Jupyter notebooks
#%pip install --upgrade ipywidgets jupyter_client

In [ ]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s {GAME}

In [ ]:
#register the model if necessary

#registry = ModelRegistry.register("", backend="openai_compatible",
                  #                                                  model_id="")
#registry.get_first_model_spec_that_unify_with("")

In [ ]:
#register the API key and url if necessary

#API_KEY = ""
#ORGANIZATION = ""
#BASE_URL ="" 

#KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

In [ ]:
def create_model(model_name: str,registry_path: str = "model_registry.json",key_path: str = "key.json", temperature: float = 0.0 ,max_tokens: int = 300) -> ChatOpenAI:
    """Returns a configured ChatOpenAI instance by looking up model_name in model_registry.json.
    Raises ValueError if the model or its required backend credentials are not found.
    SSL verification is disabled for the clp-chat model via the verify_ssl field in model_registry.json
    (add "verify_ssl": False to the model_registry.json to disable other models' verification)
    """
    with open(registry_path) as f:
        registry = json.load(f)

    entry = next((e for e in registry if e["model_name"] == model_name), None)
    if entry is None:
        raise ValueError(f"Model {model_name!r} not found in {registry_path}.")

    backend = entry["backend"]

    with open(key_path) as f:
        keys = json.load(f)

    if backend not in keys:
        raise ValueError(
            f"Backend {backend!r} (required by model {model_name!r}) "
            f"not found in {key_path}."
        )

    credentials = keys[backend]

    verify_ssl = entry.get("verify_ssl", True)
    http_client = httpx.Client(verify=verify_ssl)

    return ChatOpenAI(
        model=entry["model_id"],
        base_url=credentials["base_url"],
        api_key=credentials["api_key"],
        temperature=temperature,
        max_tokens=max_tokens,
        http_client=http_client,
    )

## 2. Agent constructor

In [ ]:
%run ./langchain_agents.ipynb

## 3. Experiment setup

In [ ]:
def make_agent(agent_type: str, model: ChatOpenAI, thread_id: str) -> ClemAgent:
    """Instantiate an agent by name."""
    if agent_type == "CoreToolsAgent":
        return CoreToolsAgent(model=model, thread_id=thread_id)
    elif agent_type == "TagExtractorAgent":
        return TagExtractorAgent(model=model, thread_id=thread_id)
    elif agent_type == "TwoToolsAgent":
        return TwoToolsAgent(model=model, thread_id=thread_id)
    elif agent_type == "LongTermPlanningAgent":
        return LongTermPlanningAgent(model=model, thread_id=thread_id)
    else:
        raise ValueError(f"Unknown agent type: {agent_type!r}.")

In [ ]:
callbacks = episode_results_folder_callbacks(
    run_dir=RUN_ID,
    result_dir_path="playpen-records",
    player_model_infos=f"{AGENT_TYPE}-{MODEL}",
)

game_env = env(GAME, single_pass=SINGLE_PASS, callbacks=callbacks)
#removed reset

print("roles:", game_env.unwrapped.game_benchmark.game_spec["roles"])

In [ ]:
model = create_model(MODEL)

roles = game_env.unwrapped.game_benchmark.game_spec["roles"]

if len(roles) == 1 or GAME == "privateshared":
    player_0 = make_agent(AGENT_TYPE, model, thread_id="player0")
    learner_agents = [player_0]
    agent_mapping = {"player_0": player_0}
else:
    player_0 = make_agent(AGENT_TYPE, model, thread_id="player0")
    player_1 = make_agent(AGENT_TYPE, model, thread_id="player1")
    learner_agents = [player_0, player_1]
    agent_mapping = {"player_0": player_0, "player_1": player_1}

print("Agent mapping:", {k: type(v).__name__ for k, v in agent_mapping.items()})

## 4. Run game

In [ ]:
import json as _json
from pathlib import Path
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def serialize_messages(msgs: list) -> list:
    """Convert LangChain messages to plain dicts for JSON serialization."""
    out = []
    for m in msgs:
        out.append({
            "type": type(m).__name__,
            "content": m.content,
            "tool_calls": getattr(m, "tool_calls", []),
        })
    return out

memory_log_dir = Path("playpen-records") / RUN_ID / GAME / "memory_logs"   #NB: kept separately from the results!
memory_log_dir.mkdir(parents=True, exist_ok=True)

all_episodes_data = []

for episode in range(NUM_EPISODES):
    game_env.reset()
    for agent in learner_agents:
        agent.reset()

    episode_memory_log = []

    context_response_pairs = []
    for step_idx, agent_id in enumerate(game_env.agent_iter()):
        context, reward, termination, truncation, info = game_env.last()
        if termination or truncation:
            response = None
        elif agent_id in agent_mapping:
            response = agent_mapping[agent_id](context)
        else:
            response = game_env.unwrapped.player_by_agent_id[agent_id](context)
        #print(f"  [obs:{agent_id}] {context['content'][:100]!r}")
        context_response_pairs.append((agent_id, context, response, reward))
        game_env.step(response)

        # snapshot memory for the agent that just acted
        agent = agent_mapping.get(agent_id)
        if agent is not None and hasattr(agent, "get_memory_snapshot"):
            msgs = agent.get_memory_snapshot()
            snapshot = {
                "step": step_idx,
                "agent_id": agent_id,
                "thread_id": agent.base_thread_id,
                "messages": serialize_messages(msgs),
            }
            episode_memory_log.append(snapshot)
            print(f"  [memory:{agent.base_thread_id}] {len(msgs)} messages: "
                  + " | ".join(f"{type(m).__name__}({m.content[:40]!r})" for m in msgs))

    # write memory log for this episode
    log_path = memory_log_dir / f"episode_{episode + 1:04d}.json"
    with open(log_path, "w") as f:
        _json.dump(episode_memory_log, f, indent=2, default=str)

    # write long-term memory
    for agent in learner_agents:
        if hasattr(agent, "get_longterm_snapshot"):
            lt_path = memory_log_dir / f"longterm_{agent.base_thread_id}.json"
            with open(lt_path, "w") as f:
                _json.dump(agent.get_longterm_snapshot(), f, indent=2, default=str)

    all_episodes_data.append(context_response_pairs)
    print(f"Episode {episode + 1}/{NUM_EPISODES} completed — {len(context_response_pairs)} steps")

print(f"\nAll episodes done. Memory logs written to: {memory_log_dir}")

In [ ]:
# Display the last episode's steps
last_episode = all_episodes_data[-1]
print(f"Last episode: {len(last_episode)} steps")
print("-" * 60)
for idx, (agent_id, context, response, reward) in enumerate(last_episode):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"  Agent({agent_id}) <- Context: {context}")
    print(f"  Agent({agent_id}) -> Response: {response}")
    print("-" * 60)

## 5. After the game

In [ ]:
results_dir = callbacks.callbacks[0].results_folder.results_dir_path
run_dir     = callbacks.callbacks[0].results_folder.run_dir
print(f"Results saved to: {results_dir}")
print(f"Run dir:          {run_dir}")
print()
print("To score results:")
print(f"  clem score -g {GAME} -r playpen-records") #or -r PATH_TO_FOLDER
print(f"  clem eval -r playpen-records")   #or -r PATH_TO_FOLDER

## 6. For debugging (this will enable the content of every LLM call)

In [ ]:
"""
from langchain_core.callbacks import BaseCallbackHandler

class MessageSpy(BaseCallbackHandler):
      def on_chat_model_start(self, serialized, messages, **kwargs):
          print("\n=== MESSAGES TO LLM ===")
          for msg in messages[0]:
              print(f"  {type(msg).__name__}: {msg.content}")
          print("======================\n")
"""

# AND add this line to the act method:
#                bla bla recursion limit = 100,
#                "callbacks": [MessageSpy()],